# Srtforge Colab Benchmark: Whisper int8_float16 + FV4

This notebook runs the benchmarked Srtforge setting used in the paper:

- FV4 vocal separation enabled
- Faster-Whisper `large-v3-turbo`
- Whisper compute type `int8_float16`
- Gemini correction disabled
- Optional WER scoring against `.srt`, `.ass`, or plain `.txt` reference text

Use a GPU runtime in Colab: `Runtime -> Change runtime type -> T4/A100/L4 GPU`.


## 1. Check GPU

The run can fall back to CPU, but the paper setting assumes CUDA. A T4 is enough for a small benchmark run, but first-time setup and FV4 processing take time.


In [ ]:
!nvidia-smi


## 2. Install Srtforge and runtime dependencies

This avoids installing the full desktop app. It installs only the Python pipeline dependencies needed for the FV4 + Faster-Whisper benchmark.


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/Kunal926/Srtforge.git"
# After you merge the branch into main, change this to "main".
SRTFORGE_REF = "codex/kunal-latest-public-clean"
REPO_DIR = Path("/content/Srtforge")

!apt-get update -qq
!apt-get install -y -qq ffmpeg
!rm -rf /content/Srtforge
!git clone --depth 1 --branch {SRTFORGE_REF} {REPO_URL} /content/Srtforge

!python -m pip install -q --upgrade pip
!python -m pip install -q \
  "numpy>=2.0.0,<3.0.0" \
  "typer>=0.12.3" "rich>=13.7.1" "tqdm>=4.66.4" "pyyaml>=6.0.1" "requests>=2.32.3" \
  "soundfile>=0.12.1" "matplotlib>=3.8.0" \
  "faster-whisper==1.2.1" "ctranslate2==4.7.1" \
  "audio-separator==0.44.1" "onnxruntime-gpu" \
  "jiwer>=3.0.0"
!python -m pip install -q -e /content/Srtforge --no-deps

os.chdir(REPO_DIR)
os.environ["SRTFORGE_PROJECT_ROOT"] = str(REPO_DIR)
print("Srtforge checkout:", REPO_DIR)


## 3. Download FV4 model assets

The notebook uses your GitHub release assets. Files are SHA-256 verified before use.


In [ ]:
import hashlib
import requests

RELEASE_BASE = "https://github.com/Kunal926/Srtforge/releases/download/v1.0.0"
MODEL_DIR = REPO_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

ASSETS = {
    "voc_fv4.ckpt": "1a9657de5fd3ed87ad4fd1a9d2069743ecb33424836973ad0f3288e2a64e90bc",
    "voc_gabox.yaml": "a4f9b0d143b5cb9d5d1d3d9414c50868107caadcac1faaf9e0f021f7bf3d1c8b",
    "download_checks.json": "d3622e1fa19c161d3cf704927711b453d593a3f1eb0f2e0838c3136907935151",
}

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def download_asset(name: str, expected_sha256: str) -> Path:
    path = MODEL_DIR / name
    if path.exists() and sha256_file(path) == expected_sha256:
        print(f"OK: {name}")
        return path
    path.unlink(missing_ok=True)
    url = f"{RELEASE_BASE}/{name}"
    print(f"Downloading {url}")
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with path.open("wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    actual = sha256_file(path)
    if actual != expected_sha256:
        path.unlink(missing_ok=True)
        raise RuntimeError(f"Bad SHA-256 for {name}: expected {expected_sha256}, got {actual}")
    print(f"OK: {name}")
    return path

for asset_name, digest in ASSETS.items():
    download_asset(asset_name, digest)

print("Model directory:", MODEL_DIR)


## 4. Provide media and optional reference

Set `MEDIA_PATH` to your `.mkv`, `.mp4`, etc. Set `REFERENCE_PATH` to a matching `.srt`, `.ass`, or `.txt` if you want WER.

For Google Drive files, uncomment the Drive mount line and use a path under `/content/drive/MyDrive/...`.


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# Option A: point to files already in Colab/Drive.
MEDIA_PATH = "/content/input.mkv"
REFERENCE_PATH = ""  # optional: /content/reference.srt, /content/reference.ass, or /content/reference.txt

# Option B: upload from your machine. Set this True, run the cell, then update MEDIA_PATH/REFERENCE_PATH.
USE_UPLOAD_WIDGET = False
if USE_UPLOAD_WIDGET:
    from google.colab import files
    uploaded = files.upload()
    print("Uploaded:", list(uploaded))

OUTPUT_DIR = Path("/content/srtforge_colab_benchmark")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Benchmark-faithful default is False. Set True only when your media has no English language tag.
ALLOW_UNTAGGED_ENGLISH = False

# ASS reference style names to score. Most authored ASS files use Default.
ASS_STYLES = {"Default"}
SDH_REFERENCE = False

media = Path(MEDIA_PATH)
if not media.exists():
    raise FileNotFoundError(f"MEDIA_PATH does not exist: {media}")
reference = Path(REFERENCE_PATH) if REFERENCE_PATH else None
if reference is not None and not reference.exists():
    raise FileNotFoundError(f"REFERENCE_PATH does not exist: {reference}")

print("Media:", media)
print("Reference:", reference or "none")
print("Output:", OUTPUT_DIR)


## 5. Run Srtforge FV4 + Whisper int8_float16

This is the benchmarked Srtforge setting: FV4 separation + Faster-Whisper `large-v3-turbo` + `int8_float16`, Gemini disabled.


In [ ]:
import json
import subprocess
import time

from srtforge.pipeline import PipelineConfig, run_pipeline
from srtforge.settings import settings

def media_duration_seconds(path: Path) -> float | None:
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        return None
    try:
        return float(result.stdout.strip())
    except ValueError:
        return None

output_srt = OUTPUT_DIR / f"{media.stem}.srt"
words_json = OUTPUT_DIR / f"{media.stem}.words.json"

config = PipelineConfig(
    media_path=media,
    output_path=output_srt,
    output_directory=None,
    temp_dir=OUTPUT_DIR / "tmp",
    models_dir=MODEL_DIR,
    fv4_model=MODEL_DIR / "voc_fv4.ckpt",
    fv4_config=MODEL_DIR / "voc_gabox.yaml",
    sample_rate=44100,
    separation_backend="fv4",
    separation_prefer_center=True,
    separation_prefer_gpu=True,
    ffmpeg_filter_chain="highpass=f=60,lowpass=f=10000,aformat=sample_fmts=flt,aresample=resampler=soxr:osf=flt:osr=16000",
    ffmpeg_extraction_mode="dual_mono_center",
    prefer_gpu=True,
    asr_engine="whisper",
    whisper_model="large-v3-turbo",
    whisper_language="en",
    whisper_compute_type="int8_float16",
    gemini_enabled=False,
    gemini_api_key=None,
    allow_untagged_english=ALLOW_UNTAGGED_ENGLISH,
    dump_word_timestamps=True,
    word_timestamps_path=words_json,
)

duration_s = media_duration_seconds(media)
started = time.perf_counter()
result = run_pipeline(config)
wall_s = time.perf_counter() - started

if result.failed:
    raise RuntimeError(result.reason)

rtf = wall_s / duration_s if duration_s else None
run_summary = {
    "media": str(media),
    "output_srt": str(result.output_path),
    "word_timestamps": str(words_json) if words_json.exists() else None,
    "wall_seconds": wall_s,
    "duration_seconds": duration_s,
    "rtf": rtf,
    "setting": "srtforge_fv4_whisper_large-v3-turbo_int8_float16",
    "gemini_enabled": False,
}
summary_path = OUTPUT_DIR / f"{media.stem}.run_summary.json"
summary_path.write_text(json.dumps(run_summary, indent=2), encoding="utf-8")

print(json.dumps(run_summary, indent=2))


## 6. Optional WER scoring

WER is computed with `jiwer.process_words`, matching the Hugging Face/Evaluate WER behavior: `(substitutions + insertions + deletions) / reference_words`.


In [ ]:
import re
from typing import Any

import jiwer

HTML_TAG_RE = re.compile(r"<[^>]+>")
ASS_TAG_RE = re.compile(r"\{[^}]*\}")
SRT_TIMESTAMP_RE = re.compile(r"(?P<start>\d{2}:\d{2}:\d{2},\d{3})\s+-->\s+(?P<end>\d{2}:\d{2}:\d{2},\d{3})")
BRACKET_RE = re.compile(r"\[[^\]]+\]|\([^)]*\)")
SPEAKER_PREFIX_RE = re.compile(r"^\s*(?:[A-Z][A-Z0-9 .'\-]{1,30}:)\s+")
PUNCT_RE = re.compile(r"[^a-z0-9\s']")
SPACE_RE = re.compile(r"\s+")

def clean_spoken_text(text: str, *, sdh: bool = False) -> str:
    text = ASS_TAG_RE.sub("", text)
    text = HTML_TAG_RE.sub("", text)
    text = text.replace("\\N", " ").replace("\\n", " ").replace("\\h", " ")
    text = text.replace("&nbsp;", " ").replace("&amp;", "&").replace("♪", " ")
    pieces = []
    for raw_line in text.splitlines() or [text]:
        line = raw_line.strip()
        if not line:
            continue
        if sdh:
            line = SPEAKER_PREFIX_RE.sub("", line)
            line = BRACKET_RE.sub("", line)
        line = SPACE_RE.sub(" ", line).strip()
        if line:
            pieces.append(line)
    return SPACE_RE.sub(" ", " ".join(pieces)).strip()

def normalize_for_wer(text: str, *, sdh: bool = False) -> str:
    text = clean_spoken_text(text, sdh=sdh).lower()
    text = text.replace("’", "'").replace("`", "'")
    text = PUNCT_RE.sub(" ", text)
    text = re.sub(r"\bo\s+f\b", "of", text)
    return SPACE_RE.sub(" ", text).strip()

def parse_srt(path: Path, *, sdh: bool = False) -> list[dict[str, Any]]:
    cues = []
    raw = path.read_text(encoding="utf-8-sig", errors="replace")
    for block in re.split(r"\n\s*\n", raw.strip()):
        lines = [line.strip("\ufeff") for line in block.splitlines() if line.strip()]
        if len(lines) < 2:
            continue
        ts_idx = next((i for i, line in enumerate(lines) if SRT_TIMESTAMP_RE.search(line)), None)
        if ts_idx is None:
            continue
        text = clean_spoken_text("\n".join(lines[ts_idx + 1:]), sdh=sdh)
        if text:
            cues.append({"text": text})
    return cues

def parse_ass(path: Path, styles: set[str], *, sdh: bool = False) -> list[dict[str, Any]]:
    cues = []
    with path.open("r", encoding="utf-8-sig", errors="replace") as f:
        for line in f:
            if not line.startswith("Dialogue:"):
                continue
            fields = line.split(":", 1)[1].lstrip().split(",", 9)
            if len(fields) < 10:
                continue
            style = fields[3].strip()
            text = fields[9]
            if style not in styles:
                continue
            cleaned = clean_spoken_text(text, sdh=sdh)
            if cleaned:
                cues.append({"text": cleaned})
    return cues

def transcript(cues: list[dict[str, Any]]) -> str:
    return " ".join(cue["text"] for cue in cues if cue.get("text"))

def load_reference_text(path: Path, *, ass_styles: set[str], sdh: bool = False) -> str:
    suffix = path.suffix.lower()
    if suffix == ".srt":
        return transcript(parse_srt(path, sdh=sdh))
    if suffix == ".ass":
        return transcript(parse_ass(path, ass_styles, sdh=sdh))
    return path.read_text(encoding="utf-8-sig", errors="replace")

def compute_wer(reference_text: str, hypothesis_text: str, *, sdh: bool = False) -> dict[str, Any]:
    ref_norm = normalize_for_wer(reference_text, sdh=sdh)
    hyp_norm = normalize_for_wer(hypothesis_text, sdh=False)
    details = jiwer.process_words(ref_norm, hyp_norm)
    return {
        "wer": details.wer,
        "wer_percent": details.wer * 100.0,
        "reference_words": len(ref_norm.split()),
        "hypothesis_words": len(hyp_norm.split()),
        "hits": details.hits,
        "substitutions": details.substitutions,
        "insertions": details.insertions,
        "deletions": details.deletions,
        "reference_normalized": ref_norm,
        "hypothesis_normalized": hyp_norm,
    }

if reference is None:
    print("No reference file provided. Skipping WER.")
else:
    reference_text = load_reference_text(reference, ass_styles=ASS_STYLES, sdh=SDH_REFERENCE)
    hypothesis_text = transcript(parse_srt(output_srt, sdh=False))
    metrics = compute_wer(reference_text, hypothesis_text, sdh=SDH_REFERENCE)
    metrics.update(run_summary)
    metrics_path = OUTPUT_DIR / f"{media.stem}.wer_metrics.json"
    metrics_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    print(f"WER: {metrics['wer_percent']:.2f}%")
    print(json.dumps({k: metrics[k] for k in ['reference_words', 'hypothesis_words', 'hits', 'substitutions', 'insertions', 'deletions', 'rtf']}, indent=2))
    print("Metrics:", metrics_path)


## 7. Download results

This downloads the generated `.srt` and JSON summaries from Colab to your machine.


In [ ]:
from google.colab import files

files.download(str(output_srt))
files.download(str(summary_path))
if reference is not None:
    files.download(str(metrics_path))


## Optional: batch benchmark helper

For multiple files, fill `BATCH_ITEMS` with `(media_path, reference_path_or_empty)` pairs and adapt the single-run cell into a loop. For long media, Colab session timeouts are the main practical limit.


In [ ]:
BATCH_ITEMS = [
    # ("/content/drive/MyDrive/show/S01E01.mkv", "/content/drive/MyDrive/show/S01E01.ass"),
]

print("Batch items configured:", len(BATCH_ITEMS))
